# Dimension Interpretation using Gemini

This notebook interprets the meaning of the 20 PCA dimensions of video embeddings.
It queries the Gemini model to analyze the 20 highest and 20 lowest scoring videos for each dimension.
To ensure the definitions are mutually exclusive and collectively exhaustive, the prompt includes the definitions of previously analyzed dimensions.

## 1) Install dependencies and set up Environment

In [ ]:
!pip install -q pandas google-generativeai

import pandas as pd
import ast
import json
import time
from pathlib import Path

try:
    import google.generativeai as genai
    from google.colab import userdata
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Configure Gemini API
    # Assumes a GOOGLE_API_KEY secret is available in Colab
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
    gemini_model = genai.GenerativeModel('gemini-1.5-flash')
    is_colab = True
except ImportError:
    print('Not running in Colab or missing genai library. Falling back to local execution and dummy definitions.')
    is_colab = False
except Exception as e:
    print(f"Error initializing Google Colab environment: {e}")
    is_colab = False

## 2) Load Data and Prepare Embeddings

We load the canonical video embeddings and expand the 20D array into separate columns (`dim_0` to `dim_19`).
We provide a local dummy data fallback so the notebook can be tested headlessly without Drive access.

In [ ]:
DATA_PATH = Path('/content/drive/MyDrive/Graphiko/exports/video_embeddings_reduced/latest/business_cluster_video_embeddings_reduced_20d.csv')

# Fallback mechanism for headless execution
if not DATA_PATH.exists():
    print(f"Warning: {DATA_PATH} not found. Creating a dummy dataset for testing.")
    
    # Create 100 dummy rows with random embeddings to allow the script to run
    import numpy as np
    dummy_data = []
    for i in range(100):
        dummy_data.append({
            'video_id': f'vid_{i}',
            'channel_name': f'Channel_{i % 5}',
            'video_title': f'Dummy Video Title {i}',
            'view_count': np.random.randint(1000, 100000),
            'embedding_20d': str([float(x) for x in np.random.randn(20)])
        })
    df = pd.DataFrame(dummy_data)
else:
    df = pd.read_csv(DATA_PATH)

dim_cols = []
if not df.empty:
    if 'embedding_20d' in df.columns:
        # Convert string representation of list to actual list if necessary
        if isinstance(df['embedding_20d'].iloc[0], str):
            df['embedding_20d'] = df['embedding_20d'].apply(ast.literal_eval)
        
        # Create columns for each dimension
        for i in range(20):
            dim_col = f'dim_{i}'
            dim_cols.append(dim_col)
            df[dim_col] = df['embedding_20d'].apply(lambda x: x[i] if len(x) > i else 0.0)

print(f"Loaded {len(df)} rows. Found {len(dim_cols)} dimension columns.")

## 3) Interpretation Algorithm
We loop through each dimension. For each one, we:
1. Sort the dataset to find the top 20 and bottom 20 videos.
2. Build a prompt including those video titles and the previously generated definitions.
3. Query Gemini for the interpretation.

In [ ]:
def call_gemini(prompt: str) -> str:
    """Calls the Gemini API if available, otherwise returns a dummy response."""
    if is_colab:
        try:
            # We add a small sleep to respect rate limits if calling in a loop
            time.sleep(2) 
            response = gemini_model.generate_content(prompt)
            return response.text.strip()
        except Exception as e:
            print(f"Error calling Gemini: {e}")
            return "Error generating definition."
    else:
        return "Dummy interpretation for headless testing."

interpretations = []
previous_definitions = ""

for dim_idx, dim_col in enumerate(dim_cols):
    if df.empty:
        break
        
    print(f"Processing {dim_col}...")
    
    # Sort and get top/bottom 20
    df_sorted = df.sort_values(by=dim_col, ascending=False)
    
    # Assuming video_title and channel_name are the main context columns
    title_col = 'video_title' if 'video_title' in df.columns else 'title'
    channel_col = 'channel_name' if 'channel_name' in df.columns else 'channelId'
    
    if title_col not in df.columns:
        # Fallback to whatever string column exists
        title_col = df.columns[0]
        
    top_20 = df_sorted.head(20)
    bottom_20 = df_sorted.tail(20)
    
    top_texts = []
    for _, row in top_20.iterrows():
        top_texts.append(f"- [{row.get(channel_col, 'Unknown')}] {row.get(title_col, 'Unknown')}")
        
    bottom_texts = []
    for _, row in bottom_20.iterrows():
        bottom_texts.append(f"- [{row.get(channel_col, 'Unknown')}] {row.get(title_col, 'Unknown')}")
        
    top_str = "\n".join(top_texts)
    bottom_str = "\n".join(bottom_texts)
    
    prompt = f"""
    You are an expert content analyst. We have extracted 20 semantic dimensions from a set of YouTube videos.
    Your task is to define what dimension {dim_idx} represents based on its highest and lowest scoring videos.

    Here are the top 20 videos that score the HIGHEST on dimension {dim_idx}:
    {top_str}

    Here are the bottom 20 videos that score the LOWEST on dimension {dim_idx}:
    {bottom_str}

    To ensure definitions are mutually exclusive and collectively exhaustive, here are the definitions of the PREVIOUSLY analyzed dimensions:
    {previous_definitions if previous_definitions else 'None yet.'}

    Based on the contrast between the highest and lowest videos, and keeping your definition distinctly different from the previously defined dimensions, provide a concise (1-2 sentences) and insightful definition for dimension {dim_idx}. 
    Focus on the semantic meaning, content type, style, or topic that this dimension captures. Do not repeat previous definitions.
    """
    
    definition = call_gemini(prompt)
    print(f"Definition {dim_idx}: {definition}\n")
    
    # Store result
    dim_data = {
        "dimension_index": dim_idx,
        "dimension_name": dim_col,
        "definition": definition,
        "top_20_sample": top_texts[:5], # Store a small sample for the artifact to keep size manageable, or all 20
        "bottom_20_sample": bottom_texts[:5]
    }
    interpretations.append(dim_data)
    
    # Update previous definitions
    previous_definitions += f"\n- Dimension {dim_idx}: {definition}"


## 4) Export Artifacts
We save the generated definitions to a JSON file at the standard Graphiko analysis path.

In [ ]:
output_root = Path('/content/drive/MyDrive/Graphiko/analysis/dimension_interpretation/')

if not is_colab or not output_root.exists():
    print("Using local directory for output fallback.")
    output_root = Path('./dimension_interpretation_output')
    
output_dir = output_root / 'latest'
output_dir.mkdir(parents=True, exist_ok=True)

artifact_path = output_dir / 'dimension_interpretations.json'

artifact = {
    "schema_version": "1.0.0",
    "artifacts": {
        "dimension_interpretations": interpretations
    },
    "run_summary": {
        "model": "gemini-1.5-flash",
        "dimensions_processed": len(interpretations),
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    }
}

with open(artifact_path, 'w') as f:
    json.dump(artifact, f, indent=2)

print(f"Exported artifact to {artifact_path}")
